# Agent Evaluation Benchmark

In [1]:
from src.config import BENCHMARK_PATH, GENERATION_MODEL
from src.agent import MAX_RETRIEVAL_K, run_agent
from src.vector_store import load_vector_store
from src.rag import create_audit_client, retrieve_clauses
from src.evaluation import load_benchmark, evaluate
from src.ollama_server import setup_ollama

## Configure the Run

In [2]:
limit = 5  # Use None for the full benchmark.
retrievals = 5  # Supported agent range: 1–10.

if limit is not None and (type(limit) is not int or limit < 1):
    raise ValueError("limit must be a positive integer or None")
if type(retrievals) is not int or not 1 <= retrievals <= MAX_RETRIEVAL_K:
    raise ValueError(f"retrievals must be an integer between 1 and {MAX_RETRIEVAL_K}")

print(f"Evaluation mode: Agent | Model: {GENERATION_MODEL} | Retrieval top-K: {retrievals}")

Evaluation mode: Agent | Model: qwen3:4b | Retrieval top-K: 5


## Load Ollama and Vector Store

In [3]:
setup_ollama()

Ollama server is already running.
nomic-embed-text is already installed.
qwen3:4b is already installed.
Required Ollama models are ready.


In [4]:
vector_store = load_vector_store()
client = create_audit_client()

## Load Benchmark

In [5]:
benchmark, requirements = load_benchmark(BENCHMARK_PATH)
print(f"Loaded {len(requirements)} requirements")
display(benchmark.head())
display(benchmark["label"].value_counts())

Loaded 50 requirements


,requirement_id,swe_id,requirement_text,clause_id,label,rationale
0,1,SWE-050,The system shall document all functional and C...,4.1.2,Meets,"Requirements, including those for COTS compone..."
1,2,SWE-051,The software requirements shall be derived fro...,4.1.3,Partial,Requirements analysis traces to systems engine...
2,3,SWE-053,Requirement changes shall be recorded in a spr...,4.1.5,Partial,"Changes are logged, satisfying basic tracking,..."
3,4,SWE-055,The system shall undergo requirements validati...,4.1.7,Meets,Validation activity confirms the software will...
4,5,SWE-057,The software architecture document shall be pr...,4.2.3,Partial,"An architecture artifact exists, but the requi..."


label
Meets      22
Partial    14
Gap        14
Name: count, dtype: int64

## Diagnose One Requirement Before the Benchmark

Restart the kernel and run the setup/loading cells first to load the updated runtime.
These diagnostic cells use only the **first** benchmark requirement, not all five.
They do not use its gold label, expected clause, or reference rationale.

First check the retriever directly. Then compare three fresh agent sessions for
that same requirement: the original 600-token budget with thinking off, a larger
2,048-token budget with thinking off, and 2,048 tokens with thinking on. The budget
and thinking overrides apply only to agent turns; the canonical audit keeps its
configured settings. Each setting can make multiple tool/model calls.

The trace prints `done_reason`, generated token count, and structured tool-call
count. `done_reason=length` indicates an output limit; `stop` with zero tool calls
shows the model finished without requesting a tool. Talking about retrieving in
plain text is not an executable tool call. No scores or artifact files are saved
by these diagnostics. Re-running these cells makes new live calls for this one
requirement. Stop here to inspect results; the five-requirement benchmark starts
in the later **Audit and Evaluate Benchmark** section.

In [12]:
# Read-only service/model inspection; this does not generate another response.
import requests
from src.config import OLLAMA_HOST

version_response = requests.get(f"{OLLAMA_HOST}/api/version", timeout=5)
version_response.raise_for_status()
model_response = requests.post(
    f"{OLLAMA_HOST}/api/show", json={"model": GENERATION_MODEL}, timeout=10,
)
model_response.raise_for_status()
model_info = model_response.json()
print("Ollama version:", version_response.json()["version"])
print("Configured model:", GENERATION_MODEL)
print("Capabilities:", model_info.get("capabilities"))
print("Template tail (inspect how the assistant response is started):")
print(model_info.get("template", "")[-360:])

Ollama version: 0.40.2
Configured model: qwen3:4b
Capabilities: ['completion', 'tools', 'thinking']
Template tail (inspect how the assistant response is started):
_call>
{"name": "{{ .Function.Name }}", "arguments": {{ .Function.Arguments }}}
</tool_call>
{{- end }}
{{- end }}{{ if not $last }}<|im_end|>
{{ end }}
{{- else if eq .Role "tool" }}<|im_start|>user
<tool_response>
{{ .Content }}
</tool_response><|im_end|>
{{ end }}
{{- if and (ne .Role "assistant") $last }}<|im_start|>assistant
<think>
{{ end }}
{{- end }}


In [10]:
diagnostic_requirement_id, diagnostic_requirement_text = requirements[0]
print(f"Diagnostic requirement: {diagnostic_requirement_id}")
print(diagnostic_requirement_text)

diagnostic_clauses = retrieve_clauses(
    diagnostic_requirement_text, vector_store, k=retrievals,
)
print(f"Direct retrieval returned {len(diagnostic_clauses)} clauses:")
print([clause.clause for clause in diagnostic_clauses])

Diagnostic requirement: 1
The system shall document all functional and COTS-derived requirements in the Software Requirements Specification (SRS), approved by the project's Configuration Control Board prior to baseline release.
Direct retrieval returned 5 clauses:
['4.1.2', '4.1.3', '2.2.8', '4.1.4', '2.1.5.12']


In [11]:
diagnostic_records = {}
for name, thinking, token_budget in [
    ("baseline", False, 600),
    ("larger-budget", False, 2048),
    ("larger-budget-thinking", True, 2048),
]:
    print(f"\n=== {name}: reasoning={thinking}, agent_num_predict={token_budget} ===", flush=True)
    diagnostic_record = run_agent(
        diagnostic_requirement_id,
        diagnostic_requirement_text,
        vector_store,
        client=client,
        k=retrievals,
        verbose=True,
        reasoning=thinking,
        agent_num_predict=token_budget,
    )
    diagnostic_records[name] = diagnostic_record
    print("Selected retrieval:", diagnostic_record.retrieved_clause_ids)
    print("Verdict:", diagnostic_record.verdict.verdict if diagnostic_record.verdict else "INVALID")
    print("Error:", diagnostic_record.error)


=== baseline: reasoning=False, agent_num_predict=600 ===
[1] Agent turn 1...
[1] Generation: done_reason=length, output_tokens=600, tool_calls=0
[1] Agent message:
Okay, let's tackle this step by step. The user wants me to audit a software requirement against NASA NPR 7150.2D. The requirement is: "The system shall document all functional and COTS-derived requirements in the Software Requirements Specification (SRS), approved by the project's Configuration Control Board prior to baseline release."

First, I need to use the retrieve_clause function to find relevant clauses in NASA NPR 7150.2D. The retrieval size is fixed at k=5. So I'll call retrieve_clause with the requirement text as the query. The function should return up to 5 matching clauses.

Wait, the user mentioned that if results are unrelated, I can search again with a focused query. But since I don't have the actual retrieved clauses here, I'll simulate the process. Let me think: NASA NPR 7150.2D is about software engineerin

### Observed Diagnostic Results

The saved diagnostics above cover requirement **1 only** on the configured
`qwen3:4b` model. Direct retrieval returned five clause IDs:
`4.1.2`, `4.1.3`, `2.2.8`, `4.1.4`, `2.1.5.12`.

| Setting | Stop reason | Generated tokens | Structured tool calls | Result |
| --- | --- | --- | --- | --- |
| Thinking off, budget 600 | `length` | 600 | 0 | INVALID |
| Thinking off, budget 2,048 | `length` | 2,048 | 0 | INVALID |
| Thinking on, budget 2,048 | `length` | 2,048 | 0 | INVALID |

**Established:** the retriever works for this requirement, but the agent never
requests it. The model exhausts both tested budgets while describing hypothetical
tool use. Increasing the budget to 2,048 did not resolve the failure.

**Possible contributor:** Ollama reports version `0.40.2` and model capabilities
`completion`, `tools`, and `thinking`. The inspected installed template appends an
opening `<think>` tag in its final-user-message branch without checking whether
thinking is enabled. In these runs, disabling thinking moved the planning text
into ordinary message content without producing a tool call. This points to
model/template and generation behavior as the next area to investigate; it does
not prove that the template alone caused the failure.

The model, installed template, and canonical audit configuration were left
unchanged. The existing five-requirement benchmark outputs were preserved, not
re-run. These single-requirement diagnostics are not benchmark accuracy results.

In [6]:
evaluation_set = benchmark.iloc[:limit].copy()
evaluation_requirements = requirements[:limit]
print(f"Selected {len(evaluation_requirements)} requirements")

Selected 5 requirements


## Audit and Evaluate Benchmark

`verbose=True` displays returned thinking, model messages, tool calls, and tool
results after each call completes (not token streaming). `reasoning=True` enables
thinking for agent turns only; the canonical audit retains its existing settings.
The model must support thinking to return it, and thinking consumes the output
budget, so this debug run can behave differently from reasoning-disabled runs.

The saved five-requirement outputs below are from your earlier run and were
not re-run during the one-requirement diagnostics above. Run this section only
when ready to start a new five-requirement benchmark.
Results remain in memory; no evaluation artifact files are saved.

In [7]:
records = []
for index, (requirement_id, requirement_text) in enumerate(evaluation_requirements, 1):
    record = run_agent(
        requirement_id,
        requirement_text,
        vector_store,
        client=client,
        k=retrievals,
        verbose=True,
        reasoning=True,
    )
    records.append(record)
    status = record.verdict.verdict if record.verdict else "FAILED"
    print(f"[{index}/{len(evaluation_requirements)}] {requirement_id}: {status}", flush=True)

[1] Agent turn 1...
[1] Thinking:
Okay, let's tackle this step by step. The user wants me to audit a software requirement against NASA NPR 7150.2D. The requirement is: "The system shall document all functional and COTS-derived requirements in the Software Requirements Specification (SRS), approved by the project's Configuration Control Board prior to baseline release."

First, I need to use the retrieve_clause function to find relevant clauses in NASA NPR 7150.2D. The retrieval size is fixed at k=5. So I'll call retrieve_clause with the requirement text as the query. The function should return up to 5 matching clauses.

Wait, the user mentioned that if results are unrelated, I can search again with a focused query. But since I don't have the actual retrieved clauses here, I'll simulate the process. Let me think: NASA NPR 7150.2D is about software engineering processes. The requirement talks about documenting functional and COTS-derived requirements in the SRS, approved by the Configura

In [8]:
rows, metrics, matrix = evaluate(evaluation_set, records, k=retrievals)

print("Evaluation mode: Agent")
print(f"Requirements evaluated: {metrics['n']}")
print(f"Verdict accuracy: {metrics['verdict_accuracy']:.1%}")
print(f"Macro F1: {metrics['macro_f1']:.4f}")
print(f"Retrieval Hit@{retrievals}: {metrics['retrieval'][f'hit_at_{retrievals}']:.1%}")
print(f"Accepted verdict rate: {metrics['schema_valid_rate']:.1%}")
print(f"Failed audits: {sum(record.verdict is None for record in records)}")

display(matrix)
display(rows)

Evaluation mode: Agent
Requirements evaluated: 5
Verdict accuracy: 0.0%
Macro F1: 0.0000
Retrieval Hit@5: 0.0%
Accepted verdict rate: 0.0%
Failed audits: 5


model_verdict,Meets,Partial,Gap,INVALID
human_verdict,,,,
Meets,0,0,0,2
Partial,0,0,0,3
Gap,0,0,0,0


,requirement_id,swe_id,requirement_text,clause_id,label,rationale,pred_verdict,pred_clause,retrieved,hit_at_1,hit_at_k,citation_member,correct
0,1,SWE-050,The system shall document all functional and C...,4.1.2,Meets,"Requirements, including those for COTS compone...",INVALID,,,False,False,False,False
1,2,SWE-051,The software requirements shall be derived fro...,4.1.3,Partial,Requirements analysis traces to systems engine...,INVALID,,,False,False,False,False
2,3,SWE-053,Requirement changes shall be recorded in a spr...,4.1.5,Partial,"Changes are logged, satisfying basic tracking,...",INVALID,,,False,False,False,False
3,4,SWE-055,The system shall undergo requirements validati...,4.1.7,Meets,Validation activity confirms the software will...,INVALID,,,False,False,False,False
4,5,SWE-057,The software architecture document shall be pr...,4.2.3,Partial,"An architecture artifact exists, but the requi...",INVALID,,,False,False,False,False


## Inspect Incorrect Predictions, Failures, and Findings

In [9]:
display(rows.loc[~rows["correct"]])

for record in records:
    if record.error:
        print(f"{record.requirement_id}: FAILED — {record.error}")
    elif record.finding_id:
        print(f"{record.requirement_id}: {record.verdict.verdict} — finding {record.finding_id}")

,requirement_id,swe_id,requirement_text,clause_id,label,rationale,pred_verdict,pred_clause,retrieved,hit_at_1,hit_at_k,citation_member,correct
0,1,SWE-050,The system shall document all functional and C...,4.1.2,Meets,"Requirements, including those for COTS compone...",INVALID,,,False,False,False,False
1,2,SWE-051,The software requirements shall be derived fro...,4.1.3,Partial,Requirements analysis traces to systems engine...,INVALID,,,False,False,False,False
2,3,SWE-053,Requirement changes shall be recorded in a spr...,4.1.5,Partial,"Changes are logged, satisfying basic tracking,...",INVALID,,,False,False,False,False
3,4,SWE-055,The system shall undergo requirements validati...,4.1.7,Meets,Validation activity confirms the software will...,INVALID,,,False,False,False,False
4,5,SWE-057,The software architecture document shall be pr...,4.2.3,Partial,"An architecture artifact exists, but the requi...",INVALID,,,False,False,False,False


1: FAILED — Agent finished without a validated assessment
2: FAILED — Agent finished without a validated assessment
3: FAILED — Agent finished without a validated assessment
4: FAILED — Agent finished without a validated assessment
5: FAILED — Agent finished without a validated assessment
